In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
# 你可把隶属度矩阵R放在一个sheet，把权重W放另一个sheet
file_path = r"your_data.xlsx"
R_sheet = "Fuzzy_R"
W_sheet = "Fuzzy_W"

R = pd.read_excel(file_path, sheet_name=R_sheet, header=None).to_numpy(dtype=float)
W = pd.read_excel(file_path, sheet_name=W_sheet, header=None).to_numpy(dtype=float).flatten()

# ========= 2) 参数模板 =========
params = {}  # 主要参数是 R 和 W

B = W @ R
level = int(np.argmax(B))
print("综合向量B=", B, "最优等级索引=", level)


# 模糊综合评价

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

模糊综合评价 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

指标边界模糊、评价等级带主观判断的问题。

## 局限性

隶属度矩阵依赖专家经验，主观性较强。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。

In [ ]:
"""
模糊综合评价

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "模糊综合评价.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
WEIGHTS = [0.4, 0.6]  # TODO: 请填写[指标权重]，说明：长度等于指标数，非负。
GRADE_NAMES = ["优", "良", "中", "差"]  # TODO: 请填写[评价等级名称]，说明：从好到差或按业务顺序。
GRADE_SCORES = [95, 80, 65, 50]  # TODO: 请填写[等级分值]，说明：长度等于等级数。
TODO_MEMBERSHIP_MATRIX = [[0.7, 0.2, 0.1, 0.0], [0.4, 0.4, 0.2, 0.0]]  # TODO: 请填写[隶属度矩阵]，说明：每行对应一个指标，每行和建议为 1。



REQUIRES_DATA = False  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    # membership_matrix 的形状为 指标数 x 评价等级数。
    membership_matrix = np.array(TODO_MEMBERSHIP_MATRIX, dtype=float)
    weights = np.array(WEIGHTS, dtype=float)
    weights = weights / weights.sum()
    grade_scores = np.array(GRADE_SCORES, dtype=float)
    fuzzy_vector = weights @ membership_matrix
    final_score = fuzzy_vector @ grade_scores
    result = pd.DataFrame({"评价等级": GRADE_NAMES, "综合隶属度": fuzzy_vector})
    result.loc[len(result)] = ["最终得分", final_score]
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
